## Centering vs. Standardizing Predictors (and their effect on VIF)

In [1]:
# --- 1. Import libraries ---
import os
import sys
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# Manually define the correct project root path based on your previous output
CORRECT_PROJECT_ROOT = "/home/rtackett/projects/Masters-level-DIY-Data-Science-Curriculum-ai-Era-/ds-zero-to-one"

# Set CWD
try:
    os.chdir(CORRECT_PROJECT_ROOT)
    print(f"✅ CWD successfully set to: {os.getcwd()}")

    # Add to sys.path for module imports (src.helpers)
    if CORRECT_PROJECT_ROOT not in sys.path:
        sys.path.append(CORRECT_PROJECT_ROOT)
        print("✅ Added project root to sys.path.")

except FileNotFoundError:
    print("❌ CRITICAL ERROR: The manually defined project path does not exist.")
    sys.exit(1)

✅ CWD successfully set to: /home/rtackett/projects/Masters-level-DIY-Data-Science-Curriculum-ai-Era-/ds-zero-to-one
✅ Added project root to sys.path.


In [3]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/processed/tips_cleaned.csv")

# numeric features only
X_raw = df[["bill_total_usd", "party_size"]]

# Standardize
scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)

# Center (subtract mean only)
X_centered = X_raw - X_raw.mean()

def compute_vif(X):
    return pd.DataFrame({
        "Feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })

print("Raw VIF:")
display(compute_vif(X_raw))

print("Standardized VIF:")
display(compute_vif(X_std))

print("Centered VIF:")
display(compute_vif(X_centered))


Raw VIF:


,Feature,VIF
0,bill_total_usd,8.684373
1,party_size,8.684373


Standardized VIF:


,Feature,VIF
0,bill_total_usd,1.557586
1,party_size,1.557586


Centered VIF:


,Feature,VIF
0,bill_total_usd,1.557586
1,party_size,1.557586


### Ridge vs. OLS Example to Show Stabilization

In [4]:
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score
import numpy as np

X = df[["bill_total_usd", "party_size"]]
y = df["tip_usd"]

ols = LinearRegression().fit(X, y)
ridge = Ridge(alpha=10).fit(X, y)

pd.DataFrame({
    "Feature": X.columns,
    "OLS Coef": ols.coef_,
    "Ridge Coef (α=10)": ridge.coef_,
})


,Feature,OLS Coef,Ridge Coef (α=10)
0,bill_total_usd,0.092713,0.093454
1,party_size,0.192598,0.180249


In [ ]:
# from src.helpers import calculate_vif

# df = pd.read_csv("data/processed/tips_cleaned.csv")

# # Convert relevant categoricals to numeric (if not already encoded)
# df_encoded = df.copy()
# df_encoded["gender_Male"] = (df_encoded["gender"].str.lower() == "male").astype(int)
# df_encoded["is_smoker_Yes"] = (df_encoded["is_smoker"].str.lower() == "yes").astype(int)

# # Choose features to test for multicollinearity
# features = ["bill_total_usd", "party_size", "gender_Male", "is_smoker_Yes"]

# # Compute VIF
# vif_table = calculate_vif(df_encoded, features)
# vif_table


In [9]:

from src.helpers import calculate_vif

features = [f for f in ["bill_total_usd", "party_size", "gender_Male", "is_smoker_Yes"]
            if f in df_encoded.columns]

print("Using features:", features)
vif_table = calculate_vif(df_encoded, features)
vif_table


Using features: ['bill_total_usd', 'party_size', 'gender_Male']


,Feature,VIF
1,bill_total_usd,1.579160
2,party_size,1.557587
3,gender_Male,1.021440


### 📘 Session 7 — Multicollinearity & VIF (Full Summary)
## 🔍 1. What Is Multicollinearity?

Multicollinearity happens when **two or more predictors are highly correlated** with each other.
This causes problems in regression:

* Coefficients become unstable
* Signs may flip randomly
* Small changes in data → big changes in β
* Standard errors inflate → low statistical power
* Hard to interpret the “true effect” of each feature

In short:

**When predictors duplicate each other’s information, regression becomes confused.**

📐 2. What Is VIF (Variance Inflation Factor)?

VIF measures how much a predictor’s variance is inflated because of collinearity.

Formula:

### Variance Inflation Factor (VIF)

The formula for VIF for predictor *i* is:

$ \text{VIF}_i = \frac{1}{1 - R_i^2} $


 = R² from regressing predictor i on all the other predictors.

VIF Interpretation Guide

| VIF      | Meaning                                      |
| -------- | -------------------------------------------- |
| **1.0**  | Predictor is perfectly independent — *great* |
| **1–5**  | Mild correlation — *generally fine*          |
| **5–10** | Moderate collinearity — *watch closely*      |
| **>10**  | Severe multicollinearity — *fix immediately* |
